# Petstore MCP Validation Notebook

Interactive companion to `docs/validation.md`.

This notebook helps you validate:
- local project health (tests and script checks)
- Petstore API connectivity
- MCP tool calls over `streamable-http`

Run cells top-to-bottom.

## 1) Configure environment variables

Set your values below before running the rest of the notebook.

In [ ]:
import os

# Update these as needed.
os.environ['PETSTORE_MCP_API_BASE_URL'] = os.environ.get('PETSTORE_MCP_API_BASE_URL', 'http://localhost:8000')
os.environ['PETSTORE_MCP_API_KEY'] = os.environ.get('PETSTORE_MCP_API_KEY', 'your-api-key')
os.environ['PETSTORE_MCP_LOG_LEVEL'] = os.environ.get('PETSTORE_MCP_LOG_LEVEL', 'DEBUG')

# MCP HTTP transport settings
os.environ['PETSTORE_MCP_TRANSPORT'] = 'streamable-http'
os.environ['PETSTORE_MCP_HOST'] = '127.0.0.1'
os.environ['PETSTORE_MCP_PORT'] = '8765'
os.environ['PETSTORE_MCP_MOUNT_PATH'] = '/'

for key in [
    'PETSTORE_MCP_API_BASE_URL',
    'PETSTORE_MCP_API_KEY',
    'PETSTORE_MCP_LOG_LEVEL',
    'PETSTORE_MCP_TRANSPORT',
    'PETSTORE_MCP_HOST',
    'PETSTORE_MCP_PORT',
    'PETSTORE_MCP_MOUNT_PATH',
]:
    print(f'{key}={os.environ.get(key)}')

## 2) Helper for shell commands

In [ ]:
import subprocess

def run(cmd: str, check: bool = True):
    print(f'\n$ {cmd}')
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=os.environ.copy())
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed ({completed.returncode}): {cmd}')
    return completed

## 3) Fast path checks

Equivalent to the recommended path in `docs/validation.md`.

In [ ]:
run('uv run pytest -q')

In [ ]:
run('uv run python scripts/validate_mcp.py --timeout 30')

## 4) Optional direct Petstore API smoke checks

In [ ]:
import httpx

base = os.environ['PETSTORE_MCP_API_BASE_URL'].rstrip('/')
headers = {}
api_key = os.environ.get('PETSTORE_MCP_API_KEY')
if api_key and api_key != 'your-api-key':
    headers['X-API-Key'] = api_key

with httpx.Client(timeout=15.0) as client:
    health = client.get(f'{base}/health', headers=headers)
    print('health', health.status_code, health.text)

    pets = client.get(f'{base}/api/v1/pet/findByStatus', params={'status': 'available'}, headers=headers)
    print('findByStatus', pets.status_code, pets.text[:500])

## 5) Start MCP server (background process)

In [ ]:
import time

mcp_server_proc = globals().get('mcp_server_proc')
if mcp_server_proc is not None and mcp_server_proc.poll() is None:
    print(f'MCP server already running (pid={mcp_server_proc.pid})')
else:
    mcp_server_proc = subprocess.Popen(
        ['uv', 'run', 'petstore-mcp'],
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    globals()['mcp_server_proc'] = mcp_server_proc
    time.sleep(2)
    print(f'Started MCP server (pid={mcp_server_proc.pid})')

## 6) MCP health and pet checks over streamable-http

In [ ]:
import asyncio
import json
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_URL = 'http://127.0.0.1:8765/mcp'

async def call_tool(tool_name: str, arguments: dict):
    async with streamablehttp_client(MCP_URL, timeout=30.0) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            if getattr(result, 'structuredContent', None) is not None:
                return result.structuredContent
            if getattr(result, 'content', None):
                text = getattr(result.content[0], 'text', None)
                if text is not None:
                    try:
                        return json.loads(text)
                    except Exception:
                        return {'text': text}
            return {'isError': getattr(result, 'isError', True)}

health = await call_tool('health_check', {'include_details': True})
pets = await call_tool('pet_find_by_status', {'status': 'available'})

print('health_check ->')
print(json.dumps(health, indent=2))
print('\npet_find_by_status ->')
print(json.dumps(pets, indent=2))

## 7) Stop MCP server (cleanup)

In [ ]:
mcp_server_proc = globals().get('mcp_server_proc')
if mcp_server_proc is None:
    print('No MCP server process found in notebook state.')
elif mcp_server_proc.poll() is not None:
    print('MCP server already stopped.')
else:
    mcp_server_proc.terminate()
    try:
        mcp_server_proc.wait(timeout=10)
    except Exception:
        mcp_server_proc.kill()
    print('MCP server stopped.')